In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
import warnings


from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)

warnings.filterwarnings('ignore')


# import warnings
# warnings.filterwarnings('ignore')


In [2]:

# Get all CSV files from the ../../data/machines folder
csv_files = glob.glob('../../data/azure_pm/machines/*.csv')

# Read each CSV file and create dataframes with the same name as the file
for file_path in csv_files:
    # Extract filename without extension
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    
    # Read CSV and assign to variable with the same name as the file
    globals()[file_name] = pd.read_csv(file_path)
    
    print(f"Loaded {file_name}.csv with shape: {globals()[file_name].shape}")

Loaded machine_1.csv with shape: (8772, 11)
Loaded machine_10.csv with shape: (8775, 11)
Loaded machine_100.csv with shape: (8766, 11)
Loaded machine_11.csv with shape: (8771, 11)
Loaded machine_12.csv with shape: (8773, 11)
Loaded machine_13.csv with shape: (8781, 11)
Loaded machine_14.csv with shape: (8772, 11)
Loaded machine_15.csv with shape: (8775, 11)
Loaded machine_16.csv with shape: (8773, 11)
Loaded machine_17.csv with shape: (8778, 11)
Loaded machine_18.csv with shape: (8770, 11)
Loaded machine_19.csv with shape: (8769, 11)
Loaded machine_2.csv with shape: (8773, 11)
Loaded machine_20.csv with shape: (8771, 11)
Loaded machine_21.csv with shape: (8773, 11)
Loaded machine_22.csv with shape: (8778, 11)
Loaded machine_23.csv with shape: (8771, 11)
Loaded machine_24.csv with shape: (8771, 11)
Loaded machine_25.csv with shape: (8777, 11)
Loaded machine_26.csv with shape: (8772, 11)
Loaded machine_27.csv with shape: (8775, 11)
Loaded machine_28.csv with shape: (8769, 11)
Loaded mach

In [3]:
# machine_1.head()
# machine_98.head()

<br> <br>

## Data Encoding and Scaling

In [4]:
def encode_and_scale_dataframe(df, target_column='failure', categorical_cols=['model'], numerical_cols=['age', 'volt', 'pressure', 'vibration'], scaling_method='minmax', exclude_features=None, remove_excluded=False):
  
    # Handle exclude_features parameter
    if exclude_features is None:
        exclude_features = []
    elif isinstance(exclude_features, str):
        exclude_features = [exclude_features]
    
    # Create a copy to avoid modifying original dataframe
    df_processed = df.copy()
    
    print("🔄 Starting encoding and scaling process...")
    print(f"📊 Original dataframe shape: {df_processed.shape}")
    
    if exclude_features:
        print(f"🚫 Excluding features from processing: {exclude_features}")
        if remove_excluded:
            print(f"🗑️  Features will be removed from output dataframe")
    
    # Dictionary to store encoders for later use
    encoders = {}
    
    # 1. ENCODE CATEGORICAL VARIABLES
    print("\n1️⃣ ENCODING CATEGORICAL VARIABLES")
    print("-" * 40)
    
    # Filter out excluded features from categorical columns
    categorical_cols_to_process = [col for col in categorical_cols if col not in exclude_features]
    excluded_categorical = [col for col in categorical_cols if col in exclude_features]
    
    if excluded_categorical:
        print(f"   🚫 Skipping categorical encoding for: {excluded_categorical}")
    
    for col in categorical_cols_to_process:
        if col in df_processed.columns:
            print(f"   Encoding '{col}'...")
            
            # Create and fit label encoder
            le = LabelEncoder()
            df_processed[col] = le.fit_transform(df_processed[col].astype(str))
            
            # Store encoder for future use
            encoders[col] = le
            
            # Display encoding mapping
            unique_values = df[col].unique()
            encoded_values = le.transform(unique_values.astype(str))
            mapping = dict(zip(unique_values, encoded_values))
            
            print(f"   ✅ {col} encoded: {mapping}")
        else:
            print(f"   ⚠️  Column '{col}' not found in dataframe")
    
    # 2. ENCODE TARGET VARIABLE (FAILURE)
    print(f"\n2️⃣ ENCODING TARGET VARIABLE: '{target_column}'")
    print("-" * 40)
    
    if target_column in exclude_features:
        print(f"   🚫 Skipping target variable encoding (excluded): '{target_column}'")
    elif target_column in df_processed.columns:
        print(f"   Encoding '{target_column}'...")
        
        # Check if target is already numeric
        if df_processed[target_column].dtype in ['object', 'category']:
            le_target = LabelEncoder()
            df_processed[target_column] = le_target.fit_transform(df_processed[target_column].astype(str))
            encoders[target_column] = le_target
            
            # Display target encoding mapping
            unique_targets = df[target_column].unique()
            encoded_targets = le_target.transform(unique_targets.astype(str))
            target_mapping = dict(zip(unique_targets, encoded_targets))
            print(f"   ✅ {target_column} encoded: {target_mapping}")
        else:
            print(f"   ℹ️  {target_column} is already numeric")
    else:
        print(f"   ⚠️  Target column '{target_column}' not found")
    
    # 3. SCALE NUMERICAL FEATURES
    print(f"\n3️⃣ SCALING NUMERICAL FEATURES ({scaling_method.upper()})")
    print("-" * 40)
    
    # Filter out excluded features from numerical columns
    numerical_cols_to_process = [col for col in numerical_cols if col not in exclude_features]
    excluded_numerical = [col for col in numerical_cols if col in exclude_features]
    
    if excluded_numerical:
        print(f"   🚫 Skipping scaling for: {excluded_numerical}")
    
    # Filter numerical columns that exist in dataframe
    existing_numerical_cols = [col for col in numerical_cols_to_process if col in df_processed.columns]
    missing_numerical_cols = [col for col in numerical_cols_to_process if col not in df_processed.columns]
    
    if missing_numerical_cols:
        print(f"   ⚠️  Missing columns: {missing_numerical_cols}")
    
    if existing_numerical_cols:
        print(f"   Scaling columns: {existing_numerical_cols}")
        
        # Choose scaler
        if scaling_method.lower() == 'standard':
            scaler = StandardScaler()
            print("   📏 Using StandardScaler (mean=0, std=1)")
        elif scaling_method.lower() == 'minmax':
            scaler = MinMaxScaler()
            print("   📏 Using MinMaxScaler (range 0-1)")
        else:
            scaler = StandardScaler()
            print("   📏 Default: Using StandardScaler")
        
        # Apply scaling
        df_processed[existing_numerical_cols] = scaler.fit_transform(df_processed[existing_numerical_cols])
        
        print("   ✅ Scaling completed")
        
        # Display scaling statistics
        print("\n   📊 SCALING STATISTICS:")
        for col in existing_numerical_cols:
            original_mean = df[col].mean()
            original_std = df[col].std()
            scaled_mean = df_processed[col].mean()
            scaled_std = df_processed[col].std()
            
            print(f"   {col:12} | Original: μ={original_mean:6.2f}, σ={original_std:6.2f} | "
                  f"Scaled: μ={scaled_mean:6.2f}, σ={scaled_std:6.2f}")
    else:
        scaler = None
        print("   ⚠️  No numerical columns to scale")
    
    # 4. REMOVE EXCLUDED FEATURES (if requested)
    if remove_excluded and exclude_features:
        print(f"\n🗑️  REMOVING EXCLUDED FEATURES")
        print("-" * 40)
        features_to_remove = [col for col in exclude_features if col in df_processed.columns]
        if features_to_remove:
            df_processed = df_processed.drop(columns=features_to_remove)
            print(f"   Removed columns: {features_to_remove}")
        else:
            print("   No excluded features found in dataframe")
    
    # 5. SUMMARY
    print(f"\n{'5️⃣' if remove_excluded else '4️⃣'} PROCESSING SUMMARY")
    print("-" * 40)
    print(f"   📊 Final dataframe shape: {df_processed.shape}")
    print(f"   🔤 Categorical columns encoded: {len([col for col in categorical_cols_to_process if col in encoders])}")
    print(f"   🔢 Numerical columns scaled: {len(existing_numerical_cols) if existing_numerical_cols else 0}")
    print(f"   🚫 Features excluded from processing: {len(exclude_features)}")
    if remove_excluded and exclude_features:
        print(f"   🗑️  Features removed from dataframe: {len([col for col in exclude_features if col in df.columns])}")
    print(f"   🎯 Target variable processed: {'Yes' if target_column in df_processed.columns and target_column not in exclude_features else 'No'}")
    
    return df_processed, encoders, scaler

In [5]:
# Create a dictionary of all machine dataframes
machine_dataframes = {}

# Get a list of variable names first to avoid iteration issues
var_names = list(globals().keys())

for var_name in var_names:
    if var_name.startswith('machine_') and isinstance(globals()[var_name], pd.DataFrame):
        machine_dataframes[var_name] = globals()[var_name]

# Apply encoding and scaling to all machine dataframes
processed_machine_dataframes = {}
encoders_dict = {}
scalers_dict = {}

print(f"Processing {len(machine_dataframes)} machine dataframes...")
print("=" * 60)

for machine_key, machine_df in machine_dataframes.items():
    print(f"\n🔧 Processing {machine_key}...")
    
    # Apply encoding and scaling
    processed_df, encoders, scaler = encode_and_scale_dataframe(
        df=machine_df,
        target_column='failure',
        categorical_cols=['errorID', 'comp'],
        numerical_cols=['volt', 'rotate', 'pressure', 'vibration'],
        scaling_method='minmax',
        exclude_features=['machineID', "age", "model"],
        remove_excluded=True, 
    )

    
    # Store processed dataframe and encoders/scalers
    processed_machine_dataframes[machine_key] = processed_df
    encoders_dict[machine_key] = encoders
    scalers_dict[machine_key] = scaler
    
    print(f"✅ {machine_key} processed successfully - Shape: {processed_df.shape}")

print(f"\n🎉 All {len(processed_machine_dataframes)} machine dataframes processed!")
print(f"📊 Total processed dataframes: {len(processed_machine_dataframes)}")

Processing 100 machine dataframes...

🔧 Processing machine_1...
🔄 Starting encoding and scaling process...
📊 Original dataframe shape: (8772, 11)
🚫 Excluding features from processing: ['machineID', 'age', 'model']
🗑️  Features will be removed from output dataframe

1️⃣ ENCODING CATEGORICAL VARIABLES
----------------------------------------
   Encoding 'errorID'...
   ✅ errorID encoded: {'0': 0, 'error1': 1, 'error3': 3, 'error5': 5, 'error4': 4, 'error2': 2}
   Encoding 'comp'...
   ✅ comp encoded: {'0': 0, 'comp4': 4, 'comp1': 1, 'comp3': 3, 'comp2': 2}

2️⃣ ENCODING TARGET VARIABLE: 'failure'
----------------------------------------
   Encoding 'failure'...
   ✅ failure encoded: {'0': 0, 'comp4': 3, 'comp1': 1, 'comp2': 2}

3️⃣ SCALING NUMERICAL FEATURES (MINMAX)
----------------------------------------
   Scaling columns: ['volt', 'rotate', 'pressure', 'vibration']
   📏 Using MinMaxScaler (range 0-1)
   ✅ Scaling completed

   📊 SCALING STATISTICS:
   volt         | Original: μ=170.

<br> <br> <br>

### Create lag features

In [6]:
## Fixed Temporal Feature Engineering with Categorical Target

def create_lag_features(df, target_col='failure', prediction_horizon=24):
    """
    Create lag features and CATEGORICAL target for time series prediction with NO future data leakage.
    
    Args:
        df: Input dataframe sorted by datetime
        target_col: Column containing failure information (categorical)
        prediction_horizon: Hours ahead to predict (default: 24 hours)
    
    Returns:
        DataFrame with lag features and proper CATEGORICAL target variable
    """
    print(f"🔄 Creating lag features and CATEGORICAL target (predicting {prediction_horizon}h ahead)...")
    print(f"Original target distribution: {df[target_col].value_counts().to_dict()}")
    
    # Ensure data is sorted by time
    df_sorted = df.sort_values('datetime').reset_index(drop=True)
    
    # Keep failure as categorical (no conversion to binary)
    print(f"Categorical failure distribution: {df_sorted[target_col].value_counts().to_dict()}")
    print(f"Unique failure types: {sorted(df_sorted[target_col].unique())}")
    
    # Create target: predict failure type within next N hours using forward-looking window
    # For categorical data, we'll take the maximum failure type in the next N hours
    df_sorted['target'] = df_sorted[target_col].rolling(window=prediction_horizon, min_periods=1).max().shift(-prediction_horizon).fillna(0)
    
    # Ensure target maintains the same data type as original failure column
    df_sorted['target'] = df_sorted['target'].astype(df_sorted[target_col].dtype)
    
    display(df_sorted["target"].value_counts())

    # Remove last N rows where we can't predict the future
    df_sorted = df_sorted.iloc[:-prediction_horizon].copy()
    
    # Create lag features for sensor data (only use PAST data)
    sensor_cols = ['volt', 'rotate', 'pressure', 'vibration']
    
    print(f"Creating lag features for sensor columns: {sensor_cols}")
    
    for col in sensor_cols:
        # Lag features (1, 6, 12, 24 hours ago)
        for lag in [1, 6, 12, 24]:
            df_sorted[f'{col}_lag_{lag}h'] = df_sorted[col].shift(lag)
        
        # Rolling statistics (past 24 hours)
        df_sorted[f'{col}_mean_24h'] = df_sorted[col].rolling(window=24, min_periods=1).mean()
        df_sorted[f'{col}_std_24h'] = df_sorted[col].rolling(window=24, min_periods=1).std()
        df_sorted[f'{col}_min_24h'] = df_sorted[col].rolling(window=24, min_periods=1).min()
        df_sorted[f'{col}_max_24h'] = df_sorted[col].rolling(window=24, min_periods=1).max()
        
        # Rolling statistics (past 6 hours)
        df_sorted[f'{col}_mean_6h'] = df_sorted[col].rolling(window=6, min_periods=1).mean()
        df_sorted[f'{col}_std_6h'] = df_sorted[col].rolling(window=6, min_periods=1).std()
    
    # Error and maintenance lag features
    df_sorted['errorID_lag_1h'] = df_sorted['errorID'].shift(1)
    df_sorted['errorID_lag_6h'] = df_sorted['errorID'].shift(6)
    df_sorted['errorID_lag_12h'] = df_sorted['errorID'].shift(12)
    df_sorted['comp_lag_1h'] = df_sorted['comp'].shift(1)
    df_sorted['comp_lag_6h'] = df_sorted['comp'].shift(6)
    df_sorted['comp_lag_12h'] = df_sorted['comp'].shift(12)
    
    # Count features (past events only)
    df_sorted['error_count_6h'] = df_sorted['errorID'].rolling(window=6, min_periods=1).sum()
    df_sorted['error_count_24h'] = df_sorted['errorID'].rolling(window=24, min_periods=1).sum()
    df_sorted['maint_count_6h'] = df_sorted['comp'].rolling(window=6, min_periods=1).sum()
    df_sorted['maint_count_24h'] = df_sorted['comp'].rolling(window=24, min_periods=1).sum()
    
    # Time features
    df_sorted['hour'] = df_sorted['datetime'].dt.hour
    df_sorted['day_of_week'] = df_sorted['datetime'].dt.dayofweek
    df_sorted['is_weekend'] = (df_sorted['datetime'].dt.dayofweek >= 5).astype(int)
    df_sorted['is_working_hours'] = ((df_sorted['hour'] >= 8) & (df_sorted['hour'] <= 17)).astype(int)
    
    # Hours since last maintenance
    maint_mask = df_sorted['comp'] > 0
    if maint_mask.any():
        last_maint_idx = -1
        hours_since_maint = []
        for i, is_maint in enumerate(maint_mask):
            if is_maint:
                last_maint_idx = i
                hours_since_maint.append(0)
            else:
                hours_since_maint.append(i - last_maint_idx if last_maint_idx >= 0 else i)
        df_sorted['hours_since_maint'] = hours_since_maint
    else:
        df_sorted['hours_since_maint'] = range(len(df_sorted))
    
    # Hours since last error
    error_mask = df_sorted['errorID'] > 0
    if error_mask.any():
        last_error_idx = -1
        hours_since_error = []
        for i, is_error in enumerate(error_mask):
            if is_error:
                last_error_idx = i
                hours_since_error.append(0)
            else:
                hours_since_error.append(i - last_error_idx if last_error_idx >= 0 else i)
        df_sorted['hours_since_error'] = hours_since_error
    else:
        df_sorted['hours_since_error'] = range(len(df_sorted))
    
    # Fill NaN values created by lag features
    df_sorted = df_sorted.fillna(0)
    
    print(f"✅ Created lag features. Final shape: {df_sorted.shape}")
    print(f"   CATEGORICAL Target distribution: {df_sorted['target'].value_counts().to_dict()}")
    print(f"   Target classes: {sorted(df_sorted['target'].unique())}")
    print(f"   Target data type: {df_sorted['target'].dtype}")
    print(f"   New feature count: {len([col for col in df_sorted.columns if col not in df.columns])}")
    
    return df_sorted


In [7]:
# Apply lag feature engineering to all processed machine dataframes
lag_machine_dataframes = {}

print(f"Creating lag features for {len(processed_machine_dataframes)} machine dataframes...")
print("=" * 70)

for machine_key, processed_df in processed_machine_dataframes.items():
    print(f"\n🔧 Processing {machine_key}...")
    
    # Get the original machine dataframe to access datetime column
    original_df = machine_dataframes[machine_key].copy()
    
    # Add datetime back to processed dataframe for lag feature creation
    processed_df_with_datetime = processed_df.copy()
    processed_df_with_datetime['datetime'] = original_df['datetime'].values
    
    # Convert datetime column to proper datetime format
    processed_df_with_datetime['datetime'] = pd.to_datetime(processed_df_with_datetime['datetime'])
    
    # Apply lag feature engineering
    df_with_lags = create_lag_features(
        df=processed_df_with_datetime,
        target_col='failure',
        prediction_horizon=24
    )
    
    # Store the result
    lag_machine_dataframes[machine_key] = df_with_lags
    
    print(f"✅ {machine_key} lag features created - Shape: {df_with_lags.shape}")

print(f"\n🎉 All {len(lag_machine_dataframes)} machine dataframes processed with lag features!")
print(f"📊 Total lag dataframes created: {len(lag_machine_dataframes)}")

# Display sample from one machine
sample_machine = 'machine_100'
if sample_machine in lag_machine_dataframes:
    print(f"\n📋 Sample from {sample_machine}:")
    print(f"Columns ({len(lag_machine_dataframes[sample_machine].columns)}): {list(lag_machine_dataframes[sample_machine].columns)}")
    print(f"Shape: {lag_machine_dataframes[sample_machine].shape}")

Creating lag features for 100 machine dataframes...

🔧 Processing machine_1...
🔄 Creating lag features and CATEGORICAL target (predicting 24h ahead)...
Original target distribution: {0: 8761, 3: 7, 2: 3, 1: 1}
Categorical failure distribution: {0: 8761, 3: 7, 2: 3, 1: 1}
Unique failure types: [0, 1, 2, 3]


0    8600
3      99
2      49
1      24
Name: target, dtype: int64

Creating lag features for sensor columns: ['volt', 'rotate', 'pressure', 'vibration']
✅ Created lag features. Final shape: (8748, 65)
   CATEGORICAL Target distribution: {0: 8576, 3: 99, 2: 49, 1: 24}
   Target classes: [0, 1, 2, 3]
   Target data type: int32
   New feature count: 57
✅ machine_1 lag features created - Shape: (8748, 65)

🔧 Processing machine_10...
🔄 Creating lag features and CATEGORICAL target (predicting 24h ahead)...
Original target distribution: {0: 8767, 2: 7, 1: 1}
Categorical failure distribution: {0: 8767, 2: 7, 1: 1}
Unique failure types: [0, 1, 2]


0    8652
2      99
1      24
Name: target, dtype: int64

Creating lag features for sensor columns: ['volt', 'rotate', 'pressure', 'vibration']
✅ Created lag features. Final shape: (8751, 65)
   CATEGORICAL Target distribution: {0: 8628, 2: 99, 1: 24}
   Target classes: [0, 1, 2]
   Target data type: int32
   New feature count: 57
✅ machine_10 lag features created - Shape: (8751, 65)

🔧 Processing machine_100...
🔄 Creating lag features and CATEGORICAL target (predicting 24h ahead)...
Original target distribution: {0: 8763, 1: 2, 2: 1}
Categorical failure distribution: {0: 8763, 1: 2, 2: 1}
Unique failure types: [0, 1, 2]


...

Creating lag features for sensor columns: ['volt', 'rotate', 'pressure', 'vibration']
✅ Created lag features. Final shape: (8750, 65)
   CATEGORICAL Target distribution: {0: 8574, 3: 101, 2: 50, 1: 25}
   Target classes: [0, 1, 2, 3]
   Target data type: int32
   New feature count: 57
✅ machine_97 lag features created - Shape: (8750, 65)

🔧 Processing machine_98...
🔄 Creating lag features and CATEGORICAL target (predicting 24h ahead)...
Original target distribution: {0: 8755, 3: 7, 1: 7, 4: 6, 2: 6}
Categorical failure distribution: {0: 8755, 3: 7, 1: 7, 4: 6, 2: 6}
Unique failure types: [0, 1, 2, 3, 4]


0    8433
3     122
4      78
1      75
2      73
Name: target, dtype: int64

Creating lag features for sensor columns: ['volt', 'rotate', 'pressure', 'vibration']
✅ Created lag features. Final shape: (8757, 65)
   CATEGORICAL Target distribution: {0: 8409, 3: 122, 4: 78, 1: 75, 2: 73}
   Target classes: [0, 1, 2, 3, 4]
   Target data type: int32
   New feature count: 57
✅ machine_98 lag features created - Shape: (8757, 65)

🔧 Processing machine_99...
🔄 Creating lag features and CATEGORICAL target (predicting 24h ahead)...
Original target distribution: {0: 8749, 3: 9, 2: 8, 4: 5, 1: 2}
Categorical failure distribution: {0: 8749, 3: 9, 2: 8, 4: 5, 1: 2}
Unique failure types: [0, 1, 2, 3, 4]


0    8315
2     146
3     144
4     120
1      48
Name: target, dtype: int64

Creating lag features for sensor columns: ['volt', 'rotate', 'pressure', 'vibration']
✅ Created lag features. Final shape: (8749, 65)
   CATEGORICAL Target distribution: {0: 8291, 2: 146, 3: 144, 4: 120, 1: 48}
   Target classes: [0, 1, 2, 3, 4]
   Target data type: int32
   New feature count: 57
✅ machine_99 lag features created - Shape: (8749, 65)

🎉 All 100 machine dataframes processed with lag features!
📊 Total lag dataframes created: 100

📋 Sample from machine_100:
Columns (65): ['datetime', 'volt', 'rotate', 'pressure', 'vibration', 'errorID', 'comp', 'failure', 'target', 'volt_lag_1h', 'volt_lag_6h', 'volt_lag_12h', 'volt_lag_24h', 'volt_mean_24h', 'volt_std_24h', 'volt_min_24h', 'volt_max_24h', 'volt_mean_6h', 'volt_std_6h', 'rotate_lag_1h', 'rotate_lag_6h', 'rotate_lag_12h', 'rotate_lag_24h', 'rotate_mean_24h', 'rotate_std_24h', 'rotate_min_24h', 'rotate_max_24h', 'rotate_mean_6h', 'rotate_std_6h', 'pressure_lag_1h', 'pressure_lag_6h', 'pressure_lag_12h', 'pressure_lag_24h', '

## Temporal Feature Engineering for Predictive Maintenance

The transformation of raw time-series sensor data into a structured format suitable for machine learning models necessitated a comprehensive temporal feature engineering approach. This process was essential to capture the temporal dependencies and historical patterns inherent in industrial machinery behavior, enabling the models to learn from past observations to predict future failure events.

### Rationale for Temporal Feature Engineering

Industrial machinery operates in complex temporal patterns where current system states are heavily influenced by historical operating conditions, maintenance activities, and error occurrences. Simple cross-sectional analysis of instantaneous sensor readings fails to capture the degradation processes, cyclical patterns, and cumulative effects that precede equipment failures. Therefore, a systematic approach to temporal feature extraction was implemented to provide predictive models with rich historical context while maintaining strict temporal causality to prevent data leakage.

### Target Variable Construction

The primary prediction task was formulated as a multi-class classification problem with a 24-hour prediction horizon. For each temporal observation, a forward-looking window of 24 hours was examined to identify the most severe failure event within this future timeframe. The target variable was constructed by taking the maximum failure severity code observed in the subsequent 24-hour period, with zero assigned to periods where no failure occurred. This approach ensures that the model learns to recognize precursor patterns that manifest up to 24 hours before actual failure events. To maintain temporal integrity, the final 24 observations of each machine's dataset were excluded from analysis, as complete future windows could not be constructed for these instances.

### Historical Sensor Feature Engineering

A multi-scale temporal feature extraction strategy was employed to capture both short-term fluctuations and longer-term trends in sensor measurements. For each primary sensor variable—voltage, rotation speed, pressure, and vibration—several categories of historical features were systematically constructed:

**Discrete Lag Features**: Direct lagged values were extracted at strategically selected intervals of 1, 6, 12, and 24 hours prior to each observation. This approach captures the immediate operational history and enables the model to identify specific temporal patterns associated with impending failures.

**Rolling Window Statistics**: To characterize the statistical behavior of sensor readings over time, comprehensive statistical metrics were computed over sliding historical windows. For each sensor, the mean, standard deviation, minimum, and maximum values were calculated over both 6-hour and 24-hour retrospective periods. These features effectively capture operational trends, variability patterns, and extreme value occurrences that may indicate developing anomalies.

### Event-Based Temporal Features

Given the discrete nature of maintenance activities and error occurrences, specialized features were engineered to capture the temporal relationships between these events and system failures. Historical values of categorical event variables, including error codes and maintenance component identifiers, were extracted at 1, 6, and 12-hour lag intervals. Additionally, cumulative event frequency features were computed by counting the total number of error occurrences and maintenance activities within rolling 6-hour and 24-hour windows, providing insight into the intensity of recent operational disturbances.

### Operational Context Features

To account for temporal patterns related to operational scheduling and human factors, several time-derived contextual features were incorporated. These included the hour of the day, day of the week, and binary indicators for weekend periods and standard working hours (08:00-17:00). Such features enable the model to account for systematic variations in operational patterns and maintenance practices that occur across different temporal cycles.

### Time-Since-Event Features

Two critical temporal distance features were engineered to quantify the elapsed time since significant operational events. The time since the last maintenance activity and the time since the last error occurrence were calculated for each observation, providing the model with explicit information about the operational state relative to these critical events. These features are particularly valuable for capturing the deterioration patterns that develop between maintenance cycles and the escalation patterns that may follow initial error manifestations.

### Missing Value Treatment

The temporal feature engineering process inherently generates missing values at the beginning of each machine's time series due to the unavailability of sufficient historical data for lag and rolling window calculations. These missing values were systematically imputed with zeros, representing a neutral baseline that does not bias the model while maintaining the temporal structure of the dataset.

This comprehensive temporal feature engineering approach transforms the raw time-series data into a rich, multi-dimensional representation that captures the complex temporal dependencies essential for accurate predictive maintenance modeling, while rigorously maintaining temporal causality to ensure the validity of predictive insights.

<br> <br> <br>

#### Save lag features to /lag_features

In [8]:
# Save lag_machine_dataframes to ../../data/azure_pm/machines/lag_features

# Create the output directory if it doesn't exist
output_dir = '../../data/azure_pm/lag_features/'
os.makedirs(output_dir, exist_ok=True)

print(f"💾 Saving {len(lag_machine_dataframes)} lag feature dataframes...")
print("=" * 60)

# Save each lag feature dataframe
for machine_key, lag_df in lag_machine_dataframes.items():
    output_path = os.path.join(output_dir, f'{machine_key}_lag_features.csv')
    
    # Save to CSV
    lag_df.to_csv(output_path, index=False)
    
    print(f"✅ Saved {machine_key} - Shape: {lag_df.shape} - File: {output_path}")

print(f"\n🎉 All lag feature dataframes saved successfully!")
print(f"📁 Output directory: {output_dir}")
print(f"📊 Total files saved: {len(lag_machine_dataframes)}")

# Display directory contents
saved_files = [f for f in os.listdir(output_dir) if f.endswith('_lag_features.csv')]
print(f"\n📋 Saved files ({len(saved_files)}):")
for i, filename in enumerate(sorted(saved_files)[:10]):  # Show first 10
    print(f"  {i+1:2d}. {filename}")
if len(saved_files) > 10:
    print(f"  ... and {len(saved_files) - 10} more files")

💾 Saving 100 lag feature dataframes...
✅ Saved machine_1 - Shape: (8748, 65) - File: ../../data/azure_pm/lag_features/machine_1_lag_features.csv
✅ Saved machine_10 - Shape: (8751, 65) - File: ../../data/azure_pm/lag_features/machine_10_lag_features.csv
✅ Saved machine_100 - Shape: (8742, 65) - File: ../../data/azure_pm/lag_features/machine_100_lag_features.csv
✅ Saved machine_11 - Shape: (8747, 65) - File: ../../data/azure_pm/lag_features/machine_11_lag_features.csv
✅ Saved machine_12 - Shape: (8749, 65) - File: ../../data/azure_pm/lag_features/machine_12_lag_features.csv
✅ Saved machine_13 - Shape: (8757, 65) - File: ../../data/azure_pm/lag_features/machine_13_lag_features.csv
✅ Saved machine_14 - Shape: (8748, 65) - File: ../../data/azure_pm/lag_features/machine_14_lag_features.csv
✅ Saved machine_15 - Shape: (8751, 65) - File: ../../data/azure_pm/lag_features/machine_15_lag_features.csv
✅ Saved machine_16 - Shape: (8749, 65) - File: ../../data/azure_pm/lag_features/machine_16_lag_fe

<br> <br> <br>

# Create balanced dataset 


#### Single Sample

- The following methodology extracts one sample (one hour) at a fixed interval (e.g., 24 hours) before each failure is a perfectly valid and common approach.

- **What it does well:** It frames the problem as a classic binary classification task. You are asking the model a very clear question: "Given the machine's state at this specific hour, what is the probability of failure in the next 24 hours?" This is interpretable and works very well with powerful models like XGBoost, LightGBM, and Random Forest, which we are using.

- **The main limitation:** It is a static snapshot. The model sees the state at T-24 hours but has no direct information about the trend leading up to that point. For instance, it doesn't know if a pressure reading has been slowly climbing for 12 hours or if it just spiked in the last hour. This crucial dynamic information is lost.


In [9]:
def rows_n_hours_before_failure(machine, machine_failure, hours):  
    failure_times = machine_failure["datetime"]

    # Convert the datetime columns to datetime if they're not already
    machine["datetime"] = pd.to_datetime(machine["datetime"])
    failure_times = pd.to_datetime(failure_times)

    # Initialize an empty list to store the rows and their indices
    rows = []
    indices = []

    # Iterate over each failure time and its index
    for idx_failure, failure_time in zip(machine_failure.index, failure_times):
        # Calculate the time n hours before the failure
        target_time = failure_time - pd.Timedelta(hours=hours)
        
        # Get the row with the closest time to the target time
        idx = (machine["datetime"] - target_time).abs().idxmin()
        closest_row = machine.loc[idx]
        
        # Append the row and the failure index to the lists
        rows.append(closest_row)
        indices.append(idx_failure)

    # Create a new dataframe with the rows
    machine_1_prev_24h = pd.DataFrame(rows)
    machine_1_prev_24h["id_failure_row"] = indices

    return machine_1_prev_24h


def update_lag_failure_target(machine_failure, machine_n_hours_before_failure, lag_failure):
    """
    Updates the 'target' variable in lag_failure with the corresponding 'failure' value from machine_failure,
    using machine_n_hours_before_failure as a bridge for the join.
    """
    if 'id_failure_row' not in machine_n_hours_before_failure.columns:
        raise ValueError("machine_n_hours_before_failure must have 'id_failure_row' column.")

    failure_values = []
    for idx in lag_failure.index:
        if idx in machine_n_hours_before_failure.index:
            id_failure_row = machine_n_hours_before_failure.loc[idx, 'id_failure_row']
            # If multiple rows, take the first
            if isinstance(id_failure_row, pd.Series):
                id_failure_row = id_failure_row.iloc[0]
            failure_val = machine_failure.loc[id_failure_row, 'failure'] if id_failure_row in machine_failure.index else None
        else:
            failure_val = None
        failure_values.append(failure_val)

    lag_failure = lag_failure.copy()
    lag_failure['target'] = failure_values
    return lag_failure


def create_safe_non_failure_samples_enhanced(machine_lag, lag_failure_updated, hours_before=24, safe_buffer_hours=48):
    """
    Create non-failure samples from 'safe zones' using the full feature set from machine_lag
    
    Parameters:
    - machine_lag: Full dataset with all engineered features
    - lag_failure_updated: Your target dataset (24h before failures)
    - hours_before: Lead time before failure (24h)
    - safe_buffer_hours: Additional buffer to ensure truly safe periods (48h recommended)
    """
    
    # Convert datetime columns to datetime if they aren't already
    machine_lag['datetime'] = pd.to_datetime(machine_lag['datetime'])
    lag_failure_updated['datetime'] = pd.to_datetime(lag_failure_updated['datetime'])
    
    # Get all failure timestamps from machine_lag where failure = 1
    failure_timestamps = machine_lag[machine_lag['failure'] == 1]['datetime']
    
    # Create exclusion zones around each failure
    exclusion_periods = []
    
    for failure_time in failure_timestamps:
        # Exclude from (failure_time - safe_buffer_hours) to failure_time
        start_exclusion = failure_time - pd.Timedelta(hours=safe_buffer_hours)
        end_exclusion = failure_time
        exclusion_periods.append((start_exclusion, end_exclusion))
    
    # Also exclude the timestamps that are already in lag_failure_updated (24h before failures)
    existing_target_timestamps = set(lag_failure_updated['datetime'])
    
    # Filter machine_lag to find safe timestamps
    safe_candidates = machine_lag.copy()
    
    # Remove rows that fall in exclusion zones
    def is_in_exclusion_zone(timestamp):
        for start_excl, end_excl in exclusion_periods:
            if start_excl <= timestamp <= end_excl:
                return True
        return False
    
    # Filter out exclusion zones and existing target timestamps
    safe_mask = (
        ~safe_candidates['datetime'].apply(is_in_exclusion_zone) &
        ~safe_candidates['datetime'].isin(existing_target_timestamps) &
        (safe_candidates['failure'] == 0)  # Only non-failure records
    )
    
    safe_candidates = safe_candidates[safe_mask]
    
    if len(safe_candidates) == 0:
        print("Warning: No safe candidates found. Consider reducing safe_buffer_hours.")
        return pd.DataFrame()
    
    # Sample non-failure records
    n_samples = min(len(lag_failure_updated), len(safe_candidates))
    
    if n_samples > len(safe_candidates):
        print(f"Warning: Only {len(safe_candidates)} safe candidates available, but {len(lag_failure_updated)} samples requested.")
        n_samples = len(safe_candidates)
    
    # Randomly sample from safe candidates
    non_failure_samples = safe_candidates.sample(n=n_samples, random_state=42).copy()
    
    # Set target to 0 for non-failure samples (they should already be 0, but ensure it)
    non_failure_samples['target'] = 0
    
    # Reset index
    non_failure_samples = non_failure_samples.reset_index(drop=True)
    
    return non_failure_samples

def create_balanced_dataset(machine_lag, lag_failure_updated, hours_before=24, safe_buffer_hours=48):
    """
    Create a balanced dataset combining failure predictions and safe non-failure samples
    """
    
    # Get non-failure samples
    non_failure_df = create_safe_non_failure_samples_enhanced(machine_lag, lag_failure_updated, hours_before, safe_buffer_hours)
    
    if len(non_failure_df) == 0:
        return lag_failure_updated.copy()
    
    # Ensure both dataframes have the same columns
    common_columns = list(set(lag_failure_updated.columns) & set(non_failure_df.columns))
    
    # If lag_failure_updated is missing some features, we need to merge them from machine_lag
    if len(common_columns) < len(machine_lag.columns):
        print("Merging additional features from machine_lag to lag_failure_updated...")
        
        # Merge lag_failure_updated with machine_lag to get all features
        lag_failure_enhanced = pd.merge(
            lag_failure_updated, 
            machine_lag, 
            on='datetime', 
            how='left',
            suffixes=('', '_from_machine_lag')
        )
        
        # Clean up duplicate columns (keep the original values from lag_failure_updated)
        for col in lag_failure_enhanced.columns:
            if col.endswith('_from_machine_lag'):
                original_col = col.replace('_from_machine_lag', '')
                if original_col in lag_failure_enhanced.columns:
                    lag_failure_enhanced[original_col] = lag_failure_enhanced[original_col].fillna(
                        lag_failure_enhanced[col]
                    )
                    lag_failure_enhanced = lag_failure_enhanced.drop(columns=[col])
        
        lag_failure_to_use = lag_failure_enhanced
    else:
        lag_failure_to_use = lag_failure_updated.copy()
    
    # Ensure all required columns are present in both dataframes
    all_required_columns = list(machine_lag.columns)
    
    # Add missing columns with appropriate default values if needed
    for col in all_required_columns:
        if col not in lag_failure_to_use.columns:
            lag_failure_to_use[col] = np.nan
        if col not in non_failure_df.columns:
            non_failure_df[col] = np.nan
    
    # Select only the required columns in the same order
    lag_failure_to_use = lag_failure_to_use[all_required_columns]
    non_failure_df = non_failure_df[all_required_columns]
    
    # Combine datasets
    balanced_dataset = pd.concat([lag_failure_to_use, non_failure_df], ignore_index=True)
    balanced_dataset = balanced_dataset.sort_values('datetime').reset_index(drop=True)
    
    return balanced_dataset, non_failure_df


# Alternative: If you want more control over sampling strategy
def create_stratified_non_failure_samples(machine_lag, lag_failure_updated, safe_buffer_hours=48, stratify_by=['comp', 'is_weekend', 'is_working_hours']):
    """
    Create stratified non-failure samples to ensure diversity across different conditions
    """
    
    machine_lag['datetime'] = pd.to_datetime(machine_lag['datetime'])
    lag_failure_updated['datetime'] = pd.to_datetime(lag_failure_updated['datetime'])
    
    # Get failure timestamps and create exclusion zones
    failure_timestamps = machine_lag[machine_lag['failure'] == 1]['datetime']
    exclusion_periods = []
    
    for failure_time in failure_timestamps:
        start_exclusion = failure_time - pd.Timedelta(hours=safe_buffer_hours)
        end_exclusion = failure_time
        exclusion_periods.append((start_exclusion, end_exclusion))
    
    # Filter safe candidates
    def is_in_exclusion_zone(timestamp):
        for start_excl, end_excl in exclusion_periods:
            if start_excl <= timestamp <= end_excl:
                return True
        return False
    
    existing_target_timestamps = set(lag_failure_updated['datetime'])
    
    safe_mask = (
        ~machine_lag['datetime'].apply(is_in_exclusion_zone) &
        ~machine_lag['datetime'].isin(existing_target_timestamps) &
        (machine_lag['failure'] == 0)
    )
    
    safe_candidates = machine_lag[safe_mask].copy()
    
    if len(safe_candidates) == 0:
        return pd.DataFrame()
    
    # Stratified sampling
    target_samples = len(lag_failure_updated)
    
    # Get the distribution of stratification variables in lag_failure_updated
    if all(col in lag_failure_updated.columns for col in stratify_by):
        # Sample proportionally to match the failure sample distribution
        stratified_samples = []
        
        for group_values, group_df in lag_failure_updated.groupby(stratify_by):
            group_size = len(group_df)
            proportion = group_size / len(lag_failure_updated)
            target_group_size = int(proportion * target_samples)
            
            # Find matching safe candidates
            mask = pd.Series(True, index=safe_candidates.index)
            for i, col in enumerate(stratify_by):
                mask &= (safe_candidates[col] == group_values[i])
            
            group_safe_candidates = safe_candidates[mask]
            
            if len(group_safe_candidates) > 0:
                sample_size = min(target_group_size, len(group_safe_candidates))
                group_sample = group_safe_candidates.sample(n=sample_size, random_state=42)
                stratified_samples.append(group_sample)
        
        if stratified_samples:
            non_failure_samples = pd.concat(stratified_samples, ignore_index=True)
        else:
            # Fallback to random sampling
            non_failure_samples = safe_candidates.sample(
                n=min(target_samples, len(safe_candidates)), 
                random_state=42
            )
    else:
        # Fallback to random sampling if stratification columns not available
        non_failure_samples = safe_candidates.sample(
            n=min(target_samples, len(safe_candidates)), 
            random_state=42
        )
    
    non_failure_samples['target'] = 0
    
    return non_failure_samples.reset_index(drop=True)

In [10]:
# Loop through machine_number 1 to 100 and generate balanced datasets for each machine

hours_before_failure = 1
safe_buffer_hours = 48

balanced_datasets = {}
non_failure_dfs = {}

for machine_number in range(1, 101):
    try:
        # Load data for this machine
        machine = pd.read_csv(f"../../data/azure_pm/machines/machine_{machine_number}.csv")
        machine_lag = pd.read_csv(f"../../data/azure_pm/lag_features/machine_{machine_number}_lag_features.csv")

        # Identify failures
        machine_failure = machine[machine['failure'] != '0']

        # Get rows n hours before each failure
        machine_n_hours_before_failure = rows_n_hours_before_failure(machine=machine,machine_failure=machine_failure,hours=hours_before_failure)
        lag_failure = rows_n_hours_before_failure(machine=machine_lag,machine_failure=machine_lag[machine_lag['failure'] != 0], hours=hours_before_failure)

        # Update lag_failure target
        lag_failure_updated = update_lag_failure_target(machine_failure=machine_failure, machine_n_hours_before_failure=machine_n_hours_before_failure, lag_failure=lag_failure)

        # Create balanced dataset
        balanced_dataset, non_failure_df = create_balanced_dataset(machine_lag=machine_lag, 
                                                                   lag_failure_updated=lag_failure_updated, 
                                                                   hours_before=hours_before_failure, 
                                                                   safe_buffer_hours=safe_buffer_hours)

        # Store results with machine number as key
        balanced_datasets[machine_number] = balanced_dataset
        non_failure_dfs[machine_number] = non_failure_df

        print(f"✅ Machine {machine_number}: Balanced dataset shape: {balanced_dataset.shape}, Non-failure shape: {non_failure_df.shape}")

    except Exception as e:
        print(f"❌ Machine {machine_number}: Error - {e}")

✅ Machine 1: Balanced dataset shape: (22, 65), Non-failure shape: (11, 65)
✅ Machine 2: Balanced dataset shape: (12, 65), Non-failure shape: (6, 65)
✅ Machine 3: Balanced dataset shape: (10, 65), Non-failure shape: (5, 65)
✅ Machine 4: Balanced dataset shape: (14, 65), Non-failure shape: (7, 65)
✅ Machine 5: Balanced dataset shape: (20, 65), Non-failure shape: (10, 65)
❌ Machine 6: Error - 'datetime'
✅ Machine 7: Balanced dataset shape: (34, 65), Non-failure shape: (17, 65)
✅ Machine 8: Balanced dataset shape: (10, 65), Non-failure shape: (5, 65)
✅ Machine 9: Balanced dataset shape: (32, 65), Non-failure shape: (16, 65)
✅ Machine 10: Balanced dataset shape: (16, 65), Non-failure shape: (8, 65)
✅ Machine 11: Balanced dataset shape: (18, 65), Non-failure shape: (9, 65)
✅ Machine 12: Balanced dataset shape: (18, 65), Non-failure shape: (9, 65)
✅ Machine 13: Balanced dataset shape: (44, 65), Non-failure shape: (22, 65)
✅ Machine 14: Balanced dataset shape: (14, 65), Non-failure shape: (7, 

In [11]:
# Join all balanced_datasets into a single DataFrame named df_union
df_union = pd.concat(
    [df for df in balanced_datasets.values() if not df.empty],
    ignore_index=True
)

# Optional: sort by datetime and reset index for consistency
df_union = df_union.sort_values(by='datetime').reset_index(drop=True)

df = df_union.copy()

# Apply mapping to target column so it's always integer
target_mapping = {
    "0": 0,
    0: 0,
    "comp1": 1,
    1: 1,
    "comp2": 2,
    2: 2,
    "comp3": 3,
    3: 3,
    "comp4": 4,
    4: 4
}
df["target"] = df["target"].map(target_mapping).astype(int)

In [12]:
df.shape

(2244, 65)

#### Dataset Construction and Balancing Methodology

To develop a robust predictive maintenance model, a meticulously constructed and balanced dataset was generated from the raw time-series sensor data. The primary objectives of this procedure were to mitigate the inherent class imbalance between failure and non-failure states and, critically, to prevent data leakage by ensuring strict temporal separation between samples representing different classes.

The process began with the identification of positive class instances. For each documented failure event across the machine population, a corresponding data snapshot was extracted from a precise temporal window of 24 hours prior to the event. This set of instances constitutes the "pre-failure" or positive class, representing the system state leading up to a malfunction.

The construction of the negative class (i.e., instances of normal, healthy operation) was performed with particular rigor to avoid look-ahead bias. First, temporal "exclusion zones" were established around each failure event. These zones spanned a 48-hour period immediately preceding each failure, effectively quarantining any data that could potentially contain early, unobserved indicators of the impending fault.

Subsequently, candidate instances for the negative class were exclusively drawn from periods of operation outside of these exclusion zones and from periods not already labeled as pre-failure. From this pool of "safe" operational data, a set of non-failure instances was randomly sampled. The number of these sampled non-failure instances was deliberately matched to the total number of pre-failure instances to create a perfectly balanced 1:1 class ratio in the final dataset. This balancing is essential to prevent the classification model from developing a bias towards the majority (non-failure) class.

Finally, the collected pre-failure (positive) and safe-operation (negative) samples from all machines were aggregated and sorted chronologically by their timestamps. The resulting dataset is thus a chronologically coherent, balanced collection of samples, specifically engineered to provide a valid basis for training and evaluating time-series classification models while minimizing the risk of data leakage.

---

#### Dataset Construction and Balancing Methodology

To develop a robust predictive maintenance model, a meticulously constructed and balanced dataset was generated from the raw time-series sensor data with the primary objectives of mitigating inherent class imbalance and preventing data leakage. This process began with the identification of positive class instances by extracting a data snapshot from the precise 24-hour window preceding each documented failure event. The corresponding negative class was constructed with particular rigor to avoid look-ahead bias; temporal "exclusion zones" spanning the 48-hour period prior to each failure were established to quarantine any data containing potential early fault indicators. Subsequently, non-failure instances were randomly sampled exclusively from these "safe" operational periods in a quantity deliberately matched to the number of pre-failure instances, thereby creating a perfectly balanced 1:1 class ratio. Finally, all collected pre-failure and safe-operation samples were aggregated and sorted chronologically by their timestamps, yielding a temporally coherent dataset specifically engineered for valid time-series model training and evaluation.


In [ ]:
df.drop(columns=[
                # 'comp_lag_1h',
                #  'comp_lag_6h', 
                #  'comp_lag_12h', 
                #  'maint_count_6h', 
                #  'maint_count_24h', 
                 'hour', 
                 'day_of_week', 
                 'is_weekend', 
                 'is_working_hours', 
                #  'hours_since_maint', 
                #  'hours_since_error',
                #  'comp'
                 ], inplace=True)
# df.columns

In [ ]:
df.head()

<br> <br> <br>

#### Key Improvements Over Your Current Approach

1. **Temporal Sequences:** Instead of just one data point 24 hours before failure, it extracts sequences (e.g., 12 hours of data) leading up to the prediction point.

2. **Advanced Feature Engineering:** Creates rich temporal features that capture:
    - **Trend analysis:** Linear slopes and momentum
    - **Volatility patterns:** Standard deviations and coefficients of variation
    - **Change dynamics:** Rate of change, acceleration, and trend shifts
    - **Temporal comparisons:** Recent vs. historical behavior
    - **Pattern detection:** Mean crossings, outliers, and error escalations

3. **Multi-scale Analysis:** Analyzes patterns at different time windows (2h, 4h, 6h, 8h, 12h) to capture both short-term and long-term dynamics.

In [13]:
import pandas as pd
import numpy as np
from typing import List, Optional, Tuple

def extract_temporal_sequences_before_failure(machine, machine_failure, hours_before=24, sequence_length=12):
    """
    Extract sequences of data for a specified period before each failure.
    
    Parameters:
    - machine: DataFrame with machine data including datetime and sensor readings
    - machine_failure: DataFrame with failure events
    - hours_before: How many hours before failure to start prediction (default: 24)
    - sequence_length: How many hours of historical data to extract (default: 12)
    
    Returns:
    - DataFrame with sequences for each failure
    """
    failure_times = pd.to_datetime(machine_failure["datetime"])
    machine["datetime"] = pd.to_datetime(machine["datetime"])
    
    sequences = []
    failure_indices = []
    
    for idx_failure, failure_time in zip(machine_failure.index, failure_times):
        # Calculate the prediction point (24 hours before failure)
        prediction_point = failure_time - pd.Timedelta(hours=hours_before)
        
        # Calculate the start of the sequence (e.g., 36 hours before failure for 12h sequence)
        sequence_start = prediction_point - pd.Timedelta(hours=sequence_length)
        
        # Extract the sequence of data
        sequence_mask = (machine["datetime"] >= sequence_start) & (machine["datetime"] <= prediction_point)
        sequence_data = machine[sequence_mask].copy()
        
        if len(sequence_data) > 0:
            # Add failure information
            sequence_data["failure_id"] = idx_failure
            sequence_data["hours_to_failure"] = (failure_time - sequence_data["datetime"]).dt.total_seconds() / 3600
            sequences.append(sequence_data)
            failure_indices.append(idx_failure)
    
    if sequences:
        combined_sequences = pd.concat(sequences, ignore_index=True)
        return combined_sequences
    else:
        return pd.DataFrame()

def create_advanced_temporal_features(sequences_df, base_sensors=['volt', 'rotate', 'pressure', 'vibration']):
    """
    Create advanced temporal features from sequences for each failure.
    
    Parameters:
    - sequences_df: DataFrame with sequences from extract_temporal_sequences_before_failure
    - base_sensors: List of sensor columns to create features for
    
    Returns:
    - DataFrame with one row per failure and advanced temporal features
    """
    if sequences_df.empty:
        return pd.DataFrame()
    
    advanced_features = []
    
    for failure_id in sequences_df['failure_id'].unique():
        failure_sequence = sequences_df[sequences_df['failure_id'] == failure_id].copy()
        failure_sequence = failure_sequence.sort_values('datetime')
        
        features = {'failure_id': failure_id}
        
        # Get the prediction point (latest timestamp in sequence)
        prediction_datetime = failure_sequence['datetime'].max()
        features['prediction_datetime'] = prediction_datetime
        
        # Basic info
        features['sequence_length'] = len(failure_sequence)
        
        for sensor in base_sensors:
            if sensor in failure_sequence.columns:
                values = failure_sequence[sensor].dropna()
                
                if len(values) > 0:
                    # Statistical features
                    features[f'{sensor}_seq_mean'] = values.mean()
                    features[f'{sensor}_seq_std'] = values.std()
                    features[f'{sensor}_seq_min'] = values.min()
                    features[f'{sensor}_seq_max'] = values.max()
                    features[f'{sensor}_seq_median'] = values.median()
                    features[f'{sensor}_seq_range'] = values.max() - values.min()
                    features[f'{sensor}_seq_iqr'] = values.quantile(0.75) - values.quantile(0.25)
                    
                    # Temporal trend features
                    if len(values) > 1:
                        # Linear trend (slope)
                        x = np.arange(len(values))
                        slope = np.polyfit(x, values, 1)[0]
                        features[f'{sensor}_seq_slope'] = slope
                        
                        # Rate of change
                        diff_values = values.diff().dropna()
                        features[f'{sensor}_seq_mean_change'] = diff_values.mean()
                        features[f'{sensor}_seq_std_change'] = diff_values.std()
                        features[f'{sensor}_seq_max_change'] = diff_values.abs().max()
                        
                        # Volatility (coefficient of variation)
                        if values.mean() != 0:
                            features[f'{sensor}_seq_cv'] = values.std() / abs(values.mean())
                        else:
                            features[f'{sensor}_seq_cv'] = 0
                        
                        # Momentum features
                        features[f'{sensor}_seq_momentum'] = values.iloc[-1] - values.iloc[0]
                        
                        # Recent vs historical comparison (last 25% vs first 75%)
                        split_point = max(1, int(0.75 * len(values)))
                        early_mean = values.iloc[:split_point].mean()
                        late_mean = values.iloc[split_point:].mean()
                        if early_mean != 0:
                            features[f'{sensor}_seq_recent_vs_early'] = (late_mean - early_mean) / abs(early_mean)
                        else:
                            features[f'{sensor}_seq_recent_vs_early'] = 0
                        
                        # Acceleration (second derivative)
                        if len(values) > 2:
                            second_diff = values.diff().diff().dropna()
                            features[f'{sensor}_seq_acceleration'] = second_diff.mean()
                        
                        # Crossing points (how many times it crosses the mean)
                        mean_val = values.mean()
                        crossings = ((values > mean_val) != (values.shift() > mean_val)).sum()
                        features[f'{sensor}_seq_mean_crossings'] = crossings
                        
                        # Outlier count (values beyond 2 standard deviations)
                        threshold = 2 * values.std()
                        outliers = abs(values - values.mean()) > threshold
                        features[f'{sensor}_seq_outlier_count'] = outliers.sum()
                        features[f'{sensor}_seq_outlier_ratio'] = outliers.mean()
        
        # Error pattern analysis
        if 'errorID' in failure_sequence.columns:
            error_sequence = failure_sequence['errorID'].dropna()
            features['error_variety'] = error_sequence.nunique()
            features['error_frequency'] = (error_sequence > 0).mean()
            features['error_escalation'] = (error_sequence.diff() > 0).sum()
        
        # Component behavior
        if 'comp' in failure_sequence.columns:
            comp_changes = (failure_sequence['comp'] != failure_sequence['comp'].shift()).sum()
            features['comp_switches'] = comp_changes
        
        # Time-based features
        features['hour_of_prediction'] = prediction_datetime.hour
        features['day_of_week'] = prediction_datetime.dayofweek
        features['is_weekend'] = prediction_datetime.dayofweek >= 5
        features['is_working_hours'] = (prediction_datetime.hour >= 8) & (prediction_datetime.hour <= 17)
        
        advanced_features.append(features)
    
    return pd.DataFrame(advanced_features)

def create_multi_scale_temporal_features(sequences_df, base_sensors=['volt', 'rotate', 'pressure', 'vibration'], 
                                       time_windows=[2, 4, 6, 8, 12]):
    """
    Create features at multiple time scales within the sequence.
    
    Parameters:
    - sequences_df: DataFrame with sequences
    - base_sensors: Sensors to analyze
    - time_windows: Different time windows to analyze (in hours from the end)
    """
    if sequences_df.empty:
        return pd.DataFrame()
    
    multi_scale_features = []
    
    for failure_id in sequences_df['failure_id'].unique():
        failure_sequence = sequences_df[sequences_df['failure_id'] == failure_id].copy()
        failure_sequence = failure_sequence.sort_values('datetime')
        
        features = {'failure_id': failure_id}
        
        for window in time_windows:
            # Get the last 'window' hours of data
            if len(failure_sequence) >= window:
                window_data = failure_sequence.tail(window)
                
                for sensor in base_sensors:
                    if sensor in window_data.columns:
                        values = window_data[sensor].dropna()
                        
                        if len(values) > 0:
                            # Key features for each time window
                            features[f'{sensor}_{window}h_mean'] = values.mean()
                            features[f'{sensor}_{window}h_std'] = values.std()
                            features[f'{sensor}_{window}h_trend'] = np.polyfit(range(len(values)), values, 1)[0] if len(values) > 1 else 0
                            features[f'{sensor}_{window}h_volatility'] = values.std() / abs(values.mean()) if values.mean() != 0 else 0
        
        # Compare different time scales
        for sensor in base_sensors:
            if sensor in failure_sequence.columns:
                # Short-term vs long-term comparison
                if f'{sensor}_2h_mean' in features and f'{sensor}_12h_mean' in features:
                    short_term = features[f'{sensor}_2h_mean']
                    long_term = features[f'{sensor}_12h_mean']
                    if long_term != 0:
                        features[f'{sensor}_short_vs_long'] = (short_term - long_term) / abs(long_term)
                    else:
                        features[f'{sensor}_short_vs_long'] = 0
                
                # Trend acceleration (difference in trends between windows)
                if f'{sensor}_2h_trend' in features and f'{sensor}_6h_trend' in features:
                    features[f'{sensor}_trend_acceleration'] = features[f'{sensor}_2h_trend'] - features[f'{sensor}_6h_trend']
        
        multi_scale_features.append(features)
    
    return pd.DataFrame(multi_scale_features)

def integrate_advanced_features_with_existing(machine_lag, machine_failure, hours_before=24, sequence_length=12):
    """
    Main function to integrate advanced temporal features with your existing approach.
    
    Parameters:
    - machine_lag: Your existing dataset with engineered features
    - machine_failure: Failure events dataset
    - hours_before: Hours before failure for prediction
    - sequence_length: Length of sequence to analyze
    
    Returns:
    - Enhanced dataset with advanced temporal features
    """
    print("Extracting temporal sequences...")
    sequences = extract_temporal_sequences_before_failure(
        machine_lag, machine_failure, hours_before, sequence_length
    )
    
    if sequences.empty:
        print("No sequences extracted. Returning original data.")
        return machine_lag
    
    print("Creating advanced temporal features...")
    advanced_features = create_advanced_temporal_features(sequences)
    
    print("Creating multi-scale temporal features...")
    multi_scale_features = create_multi_scale_temporal_features(sequences)
    
    # Merge the advanced features
    if not advanced_features.empty and not multi_scale_features.empty:
        enhanced_features = pd.merge(advanced_features, multi_scale_features, on='failure_id', how='outer')
    elif not advanced_features.empty:
        enhanced_features = advanced_features
    elif not multi_scale_features.empty:
        enhanced_features = multi_scale_features
    else:
        print("No advanced features created. Returning original data.")
        return machine_lag
    
    print("Integrating with existing dataset...")
    # Get your existing failure predictions using your original method
    existing_failure_predictions = rows_n_hours_before_failure(machine_lag, machine_failure, hours_before)
    
    # Add failure_id to match with advanced features
    existing_failure_predictions['failure_id'] = existing_failure_predictions['id_failure_row']
    
    # Merge advanced features with existing predictions
    enhanced_dataset = pd.merge(
        existing_failure_predictions, 
        enhanced_features, 
        on='failure_id', 
        how='left'
    )
    
    # Set target variable
    enhanced_dataset['target'] = 1  # These are all failure predictions
    
    print(f"Enhanced dataset created with {len(enhanced_dataset)} failure samples and {enhanced_features.shape[1]-1} new advanced features.")
    
    return enhanced_dataset

def create_enhanced_balanced_dataset(machine_lag, machine_failure, hours_before=24, sequence_length=12, safe_buffer_hours=48):
    """
    Create a balanced dataset using advanced temporal features.
    """
    # Get enhanced failure predictions
    enhanced_failure_data = integrate_advanced_features_with_existing(
        machine_lag, machine_failure, hours_before, sequence_length
    )
    
    if enhanced_failure_data.empty:
        return pd.DataFrame(), pd.DataFrame()
    
    # Create non-failure samples using existing method
    print("Creating non-failure samples...")
    non_failure_samples = create_safe_non_failure_samples_enhanced(
        machine_lag, enhanced_failure_data, hours_before, safe_buffer_hours
    )
    
    if non_failure_samples.empty:
        return enhanced_failure_data, pd.DataFrame()
    
    # For non-failure samples, we need to create temporal features as well
    print("Creating temporal features for non-failure samples...")
    non_failure_enhanced = []
    
    for idx, row in non_failure_samples.iterrows():
        sample_time = row['datetime']
        
        # Extract sequence for this non-failure sample
        sequence_start = sample_time - pd.Timedelta(hours=sequence_length)
        sequence_mask = (machine_lag["datetime"] >= sequence_start) & (machine_lag["datetime"] <= sample_time)
        sequence_data = machine_lag[sequence_mask].copy()
        
        if len(sequence_data) > 0:
            # Create a temporary failure ID for processing
            sequence_data['failure_id'] = f'non_failure_{idx}'
            
            # Create advanced features for this sequence
            temp_df = pd.DataFrame([sequence_data.to_dict('records')])
            temp_df = temp_df.explode(list(temp_df.columns)).reset_index(drop=True)
            
            # This is a simplified version - you might want to implement full feature creation
            # For now, we'll use the basic statistical features
            features = {'failure_id': f'non_failure_{idx}'}
            
            base_sensors = ['volt', 'rotate', 'pressure', 'vibration']
            for sensor in base_sensors:
                if sensor in sequence_data.columns:
                    values = sequence_data[sensor].dropna()
                    if len(values) > 0:
                        features[f'{sensor}_seq_mean'] = values.mean()
                        features[f'{sensor}_seq_std'] = values.std()
                        features[f'{sensor}_seq_slope'] = np.polyfit(range(len(values)), values, 1)[0] if len(values) > 1 else 0
            
            # Combine with original row data
            enhanced_row = {**row.to_dict(), **features}
            enhanced_row['target'] = 0
            non_failure_enhanced.append(enhanced_row)
    
    if non_failure_enhanced:
        non_failure_enhanced_df = pd.DataFrame(non_failure_enhanced)
        
        # Combine failure and non-failure samples
        # Align columns
        failure_cols = set(enhanced_failure_data.columns)
        non_failure_cols = set(non_failure_enhanced_df.columns)
        common_cols = failure_cols.intersection(non_failure_cols)
        
        balanced_dataset = pd.concat([
            enhanced_failure_data[list(common_cols)],
            non_failure_enhanced_df[list(common_cols)]
        ], ignore_index=True)
        
        balanced_dataset = balanced_dataset.sort_values('datetime').reset_index(drop=True)
        
        print(f"Balanced dataset created with {len(enhanced_failure_data)} failure samples and {len(non_failure_enhanced_df)} non-failure samples.")
        
        return balanced_dataset, non_failure_enhanced_df
    else:
        return enhanced_failure_data, pd.DataFrame()

# Example usage and feature importance analysis
def analyze_advanced_features(enhanced_dataset, target_col='target'):
    """
    Analyze the importance and distribution of advanced features.
    """
    print("Advanced Feature Analysis")
    print("=" * 50)
    
    # Identify advanced features (those with 'seq' in the name)
    advanced_feature_cols = [col for col in enhanced_dataset.columns if 'seq' in col or any(window in col for window in ['2h_', '4h_', '6h_', '8h_', '12h_'])]
    
    print(f"Number of advanced temporal features: {len(advanced_feature_cols)}")
    print("\nAdvanced features created:")
    for feature in advanced_feature_cols:
        print(f"  - {feature}")
    
    # Basic statistics
    if target_col in enhanced_dataset.columns:
        print(f"\nDataset composition:")
        print(f"  - Failure samples: {(enhanced_dataset[target_col] == 1).sum()}")
        print(f"  - Non-failure samples: {(enhanced_dataset[target_col] == 0).sum()}")
        
        # Feature correlation with target
        numeric_cols = enhanced_dataset.select_dtypes(include=[np.number]).columns
        correlations = enhanced_dataset[numeric_cols].corrwith(enhanced_dataset[target_col]).abs().sort_values(ascending=False)
        
        print("\nTop 10 features correlated with target:")
        for i, (feature, corr) in enumerate(correlations.head(10).items()):
            print(f"  {i+1}. {feature}: {corr:.4f}")
    
    return advanced_feature_cols



# ============================================================================================================================================

def extract_temporal_sequences_before_failure_fixed_count(machine, machine_failure, hours_before=24, sequence_count=12):
    """
    Extract a fixed number of data points before each failure.
    
    Parameters:
    - machine: DataFrame with machine data including datetime and sensor readings
    - machine_failure: DataFrame with failure events
    - hours_before: How many hours before failure to start prediction (default: 24)
    - sequence_count: How many data points to extract before the prediction point (default: 12)
    
    Returns:
    - DataFrame with sequences for each failure
    """
    failure_times = pd.to_datetime(machine_failure["datetime"])
    machine["datetime"] = pd.to_datetime(machine["datetime"])
    
    # Sort machine data by datetime
    machine_sorted = machine.sort_values('datetime').reset_index(drop=True)
    
    sequences = []
    failure_indices = []
    
    for idx_failure, failure_time in zip(machine_failure.index, failure_times):
        # Calculate the prediction point (hours_before failure)
        prediction_point = failure_time - pd.Timedelta(hours=hours_before)
        
        # Find the closest data point to prediction_point
        time_diffs = abs(machine_sorted["datetime"] - prediction_point)
        prediction_idx = time_diffs.idxmin()
        
        # Extract the specified number of data points before the prediction point
        start_idx = max(0, prediction_idx - sequence_count + 1)
        end_idx = prediction_idx + 1
        
        sequence_data = machine_sorted.iloc[start_idx:end_idx].copy()
        
        if len(sequence_data) > 0:
            # Add failure information
            sequence_data["failure_id"] = idx_failure
            sequence_data["hours_to_failure"] = (failure_time - sequence_data["datetime"]).dt.total_seconds() / 3600
            sequence_data["sequence_position"] = range(len(sequence_data))  # Position within sequence
            sequences.append(sequence_data)
            failure_indices.append(idx_failure)
            
            print(f"Failure {idx_failure}: Extracted {len(sequence_data)} data points")
    
    if sequences:
        combined_sequences = pd.concat(sequences, ignore_index=True)
        return combined_sequences
    else:
        return pd.DataFrame()


# ============================================================================================================================================

# Alternative: Time-based with Minimum Count
# If you want to ensure you get a minimum number of data points within a time window:

def extract_temporal_sequences_with_min_count(machine, machine_failure, hours_before=24, sequence_length_hours=12, min_data_points=6):
    """
    Extract sequences with a minimum number of data points within a time window.
    """
    failure_times = pd.to_datetime(machine_failure["datetime"])
    machine["datetime"] = pd.to_datetime(machine["datetime"])
    machine_sorted = machine.sort_values('datetime').reset_index(drop=True)
    
    sequences = []
    
    for idx_failure, failure_time in zip(machine_failure.index, failure_times):
        prediction_point = failure_time - pd.Timedelta(hours=hours_before)
        sequence_start = prediction_point - pd.Timedelta(hours=sequence_length_hours)
        
        # Extract data within time window
        time_mask = (machine_sorted["datetime"] >= sequence_start) & (machine_sorted["datetime"] <= prediction_point)
        sequence_data = machine_sorted[time_mask].copy()
        
        # If we don't have enough data points, expand the window
        if len(sequence_data) < min_data_points:
            # Find closest data point to prediction_point
            time_diffs = abs(machine_sorted["datetime"] - prediction_point)
            prediction_idx = time_diffs.idxmin()
            
            # Take the last min_data_points before prediction point
            start_idx = max(0, prediction_idx - min_data_points + 1)
            end_idx = prediction_idx + 1
            sequence_data = machine_sorted.iloc[start_idx:end_idx].copy()
        
        if len(sequence_data) > 0:
            sequence_data["failure_id"] = idx_failure
            sequence_data["hours_to_failure"] = (failure_time - sequence_data["datetime"]).dt.total_seconds() / 3600
            sequences.append(sequence_data)
            
            print(f"Failure {idx_failure}: Extracted {len(sequence_data)} data points")
    
    return pd.concat(sequences, ignore_index=True) if sequences else pd.DataFrame()

# ============================================================================================================================================

# Update Your Main Function
# Replace your current function call with the new one:

def integrate_advanced_features_with_existing_fixed_count(machine_lag, machine_failure, hours_before=24, sequence_count=12):
    """
    Main function to integrate advanced temporal features with fixed number of data points.
    """
    print(f"Extracting {sequence_count} data points before each failure...")
    sequences = extract_temporal_sequences_before_failure_fixed_count(
        machine_lag, machine_failure, hours_before, sequence_count
    )
    
    if sequences.empty:
        print("No sequences extracted. Returning original data.")
        return pd.DataFrame()
    
    print(f"Total sequences extracted: {len(sequences)}")
    print(f"Unique failures: {sequences['failure_id'].nunique()}")
    
    # Continue with your existing feature creation logic...
    print("Creating advanced temporal features...")
    advanced_features = create_advanced_temporal_features(sequences)
    
    print("Creating multi-scale temporal features...")
    multi_scale_features = create_multi_scale_temporal_features(sequences)
    
    # Rest of your existing code...
    if not advanced_features.empty and not multi_scale_features.empty:
        enhanced_features = pd.merge(advanced_features, multi_scale_features, on='failure_id', how='outer')
    elif not advanced_features.empty:
        enhanced_features = advanced_features
    elif not multi_scale_features.empty:
        enhanced_features = multi_scale_features
    else:
        print("No advanced features created.")
        return pd.DataFrame()
    
    # Get existing failure predictions
    existing_failure_predictions = rows_n_hours_before_failure(machine_lag, machine_failure, hours_before)
    existing_failure_predictions['failure_id'] = existing_failure_predictions['id_failure_row']
    
    # Merge advanced features
    enhanced_dataset = pd.merge(
        existing_failure_predictions, 
        enhanced_features, 
        on='failure_id', 
        how='left'
    )
    
    enhanced_dataset['target'] = 1
    
    print(f"Enhanced dataset created with {len(enhanced_dataset)} failure samples and {enhanced_features.shape[1]-1} new advanced features.")
    
    return enhanced_dataset



In [16]:
# Step 1: Test with a single machine first
machine_number = 1


# Load the specific machine data
machine = pd.read_csv(f"../../data/azure_pm/machines/machine_{machine_number}.csv")
machine_lag = pd.read_csv(f"../../data/azure_pm/lag_features/machine_{machine_number}_lag_features.csv")

# Identify failures for this machine
machine_failure = machine[machine['failure'] != '0']

print(f"Testing advanced features on machine_{machine_number}")
print(f"Machine data shape: {machine.shape}")
print(f"Machine lag data shape: {machine_lag.shape}")
print(f"Number of failures: {len(machine_failure)}")

# Step 2: Create enhanced failure predictions with temporal features
# enhanced_failure_data = integrate_advanced_features_with_existing(
#     machine_lag=machine_lag,
#     machine_failure=machine_failure,
#     hours_before=24,  # Same as your current approach
#     sequence_length=12  # 12 hours of historical data
# )

# Updated test with fixed sequence count
enhanced_failure_data = integrate_advanced_features_with_existing_fixed_count(
    machine_lag=machine_lag,
    machine_failure=machine_failure,
    hours_before=24,
    sequence_count=24  # This will extract exactly 12 data points before each failure
)



# Step 3: Create a balanced dataset with advanced features
balanced_dataset, non_failure_enhanced = create_enhanced_balanced_dataset(
    machine_lag=machine_lag,
    machine_failure=machine_failure,
    hours_before=24,
    sequence_length=12,
    safe_buffer_hours=48
)

# Step 4: Analyze the new features
if not balanced_dataset.empty:
    advanced_feature_cols = analyze_advanced_features(balanced_dataset)
    print(f"\nBalanced dataset shape: {balanced_dataset.shape}")
    print(f"Advanced features created: {len(advanced_feature_cols)}")
else:
    print("No balanced dataset created")


    

Testing advanced features on machine_1
Machine data shape: (8772, 11)
Machine lag data shape: (8748, 65)
Number of failures: 11
Extracting 24 data points before each failure...
Failure 96: Extracted 24 data points
Failure 97: Extracted 24 data points
Failure 1539: Extracted 24 data points
Failure 2620: Extracted 24 data points
Failure 4061: Extracted 24 data points
Failure 4062: Extracted 24 data points
Failure 5862: Extracted 24 data points
Failure 5863: Extracted 24 data points
Failure 6945: Extracted 24 data points
Failure 6946: Extracted 24 data points
Failure 8387: Extracted 24 data points
Total sequences extracted: 264
Unique failures: 11
Creating advanced temporal features...
Creating multi-scale temporal features...
Enhanced dataset created with 11 failure samples and 170 new advanced features.
Extracting temporal sequences...
Creating advanced temporal features...
Creating multi-scale temporal features...
Integrating with existing dataset...
Enhanced dataset created with 11 fa

In [17]:
enhanced_failure_data

,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,vibration_12h_trend,vibration_12h_volatility,volt_short_vs_long,volt_trend_acceleration,rotate_short_vs_long,rotate_trend_acceleration,pressure_short_vs_long,pressure_trend_acceleration,vibration_short_vs_long,vibration_trend_acceleration
0,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,1,0.433966,...,0.003507,0.195595,-0.134912,-0.047451,0.059646,-0.020432,-0.061225,-0.022002,0.013671,-0.191548
1,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,1,0.433966,...,0.003507,0.195595,-0.134912,-0.047451,0.059646,-0.020432,-0.061225,-0.022002,0.013671,-0.191548
2,2015-03-05 06:00:00,0.754059,0.400178,0.465682,0.442350,1,0,0,1,0.606900,...,-0.000110,0.291600,0.177548,0.075958,-0.151012,-0.063616,-0.076240,0.105180,-0.002366,0.160017
3,2015-04-19 06:00:00,0.317181,0.392709,0.369041,0.328448,2,0,0,1,0.476150,...,-0.009631,0.206165,-0.144837,-0.157793,0.136445,-0.038431,-0.192896,0.029703,-0.008532,-0.147468
4,2015-06-18 06:00:00,0.406874,0.540621,0.396114,0.632441,5,0,0,1,0.535453,...,0.002585,0.118051,0.108565,-0.150656,-0.057579,-0.033906,-0.096890,-0.016890,-0.003737,-0.048434
5,2015-06-18 06:00:00,0.406874,0.540621,0.396114,0.632441,5,0,0,1,0.535453,...,0.002585,0.118051,0.108565,-0.150656,-0.057579,-0.033906,-0.096890,-0.016890,-0.003737,-0.048434
6,2015-09-01 06:00:00,0.583832,0.517306,0.652636,0.523186,5,0,0,1,0.543068,...,-0.019992,0.189346,0.330663,0.006136,-0.099765,0.063128,0.096707,0.247424,-0.178145,0.070515
7,2015-09-01 06:00:00,0.583832,0.517306,0.652636,0.523186,5,0,0,1,0.543068,...,-0.019992,0.189346,0.330663,0.006136,-0.099765,0.063128,0.096707,0.247424,-0.178145,0.070515
8,2015-10-16 06:00:00,0.707202,0.211898,0.442377,0.381762,3,0,0,1,0.410665,...,-0.001505,0.299857,0.162065,0.288581,-0.430987,0.073279,0.129780,-0.021514,0.205501,-0.047834
9,2015-10-16 06:00:00,0.707202,0.211898,0.442377,0.381762,3,0,0,1,0.410665,...,-0.001505,0.299857,0.162065,0.288581,-0.430987,0.073279,0.129780,-0.021514,0.205501,-0.047834


In [32]:
# Enhanced version of your machine processing loop - EXCLUDING machine 6 and 77
enhanced_balanced_datasets = {}
enhanced_non_failure_dfs = {}

# Create list of machine numbers excluding 6 and 77
machine_numbers = [i for i in range(1, 101) if i not in [6, 77]]

for machine_number in machine_numbers:
    try:
        # Load data for this machine
        machine = pd.read_csv(f"../../data/azure_pm/machines/machine_{machine_number}.csv")
        machine_lag = pd.read_csv(f"../../data/azure_pm/lag_features/machine_{machine_number}_lag_features.csv")

        # Identify failures
        machine_failure = machine[machine['failure'] != '0']
        
        if len(machine_failure) == 0:
            print(f"⚠️  Machine {machine_number}: No failures found, skipping")
            continue

        print(f"🔧 Processing machine {machine_number} with {len(machine_failure)} failures...")

        # Create enhanced balanced dataset with advanced temporal features
        balanced_dataset, non_failure_enhanced = create_enhanced_balanced_dataset(
            machine_lag=machine_lag,
            machine_failure=machine_failure,
            hours_before=24,
            sequence_length=6,
            safe_buffer_hours=48
        )

        if not balanced_dataset.empty:
            # Store results with machine number as key
            enhanced_balanced_datasets[machine_number] = balanced_dataset
            enhanced_non_failure_dfs[machine_number] = non_failure_enhanced

            print(f"✅ Machine {machine_number}: Enhanced dataset shape: {balanced_dataset.shape}")
        else:
            print(f"⚠️  Machine {machine_number}: No enhanced dataset created")

    except Exception as e:
        print(f"❌ Machine {machine_number}: Error - {e}")

# Create union of all enhanced datasets
if enhanced_balanced_datasets:
    df_union_enhanced = pd.concat(
        [df for df in enhanced_balanced_datasets.values() if not df.empty],
        ignore_index=True
    )
    
    # Sort by datetime and reset index for consistency
    df_union_enhanced = df_union_enhanced.sort_values(by='datetime').reset_index(drop=True)
    
    print(f"\n🎉 Enhanced union dataset created with shape: {df_union_enhanced.shape}")
    print(f"📊 Processed {len(enhanced_balanced_datasets)} machines (excluding machines 6 and 77)")
    
    # Analyze the advanced features
    advanced_feature_cols = analyze_advanced_features(df_union_enhanced)
    
else:
    print("❌ No enhanced datasets were created")

🔧 Processing machine 1 with 11 failures...
Extracting temporal sequences...
Creating advanced temporal features...
Creating multi-scale temporal features...
Integrating with existing dataset...
Enhanced dataset created with 11 failure samples and 150 new advanced features.
Creating non-failure samples...
Creating temporal features for non-failure samples...
Balanced dataset created with 11 failure samples and 11 non-failure samples.
✅ Machine 1: Enhanced dataset shape: (22, 75)
🔧 Processing machine 2 with 6 failures...
Extracting temporal sequences...
Creating advanced temporal features...
Creating multi-scale temporal features...
Integrating with existing dataset...
Enhanced dataset created with 6 failure samples and 150 new advanced features.
Creating non-failure samples...
Creating temporal features for non-failure samples...
Balanced dataset created with 6 failure samples and 6 non-failure samples.
✅ Machine 2: Enhanced dataset shape: (12, 75)
🔧 Processing machine 3 with 5 failures

In [34]:
# To see your final datasets:
print(f"Standard dataset (df) shape: {df.shape}")
print(f"Enhanced dataset (df_union_enhanced) shape: {df_union_enhanced.shape}")

# Compare feature counts
print(f"Standard dataset features: {len(df.columns)}")
print(f"Enhanced dataset features: {len(df_union_enhanced.columns)}")

# Show target distribution
print(f"Standard dataset target distribution:\n{df['target'].value_counts()}")
print(f"Enhanced dataset target distribution:\n{df_union_enhanced['target'].value_counts()}")

Standard dataset (df) shape: (2244, 61)
Enhanced dataset (df_union_enhanced) shape: (2244, 75)
Standard dataset features: 61
Enhanced dataset features: 75
Standard dataset target distribution:
0    1122
2     404
1     330
4     207
3     181
Name: target, dtype: int64
Enhanced dataset target distribution:
1    1122
0    1122
Name: target, dtype: int64


In [35]:
df_union_enhanced


,errorID,pressure_seq_std,rotate_lag_24h,hour,pressure_lag_12h,hours_since_error,rotate_lag_1h,pressure,pressure_lag_6h,pressure_std_6h,...,volt_lag_6h,volt_seq_std,volt_max_24h,rotate_max_24h,maint_count_24h,rotate_mean_24h,pressure_min_24h,vibration_seq_mean,vibration_std_6h,vibration_lag_24h
0,0,NaN,0.000000,6,0.000000,0,0.000000,0.186904,0.000000,0.000000,...,0.000000,NaN,0.428164,0.568321,0.0,0.568321,0.186904,NaN,0.000000,0.000000
1,0,NaN,0.000000,6,0.000000,0,0.000000,0.400767,0.000000,0.000000,...,0.000000,NaN,0.729568,0.545589,0.0,0.545589,0.400767,NaN,0.000000,0.000000
2,0,NaN,0.000000,6,0.000000,0,0.000000,0.211396,0.000000,0.000000,...,0.000000,NaN,0.658861,0.711585,0.0,0.711585,0.211396,NaN,0.000000,0.000000
3,0,NaN,0.000000,6,0.000000,0,0.000000,0.280819,0.000000,0.000000,...,0.000000,NaN,0.409183,0.577518,4.0,0.577518,0.280819,NaN,0.000000,0.000000
4,0,NaN,0.000000,6,0.000000,0,0.000000,0.339030,0.000000,0.000000,...,0.000000,NaN,0.753752,0.527722,0.0,0.527722,0.339030,NaN,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2239,1,0.083634,0.558630,6,0.507952,0,0.466101,0.252608,0.207316,0.063806,...,0.672530,0.146222,0.867515,0.686824,0.0,0.496986,0.187443,0.340708,0.143712,0.520549
2240,1,0.083634,0.558630,6,0.507952,0,0.466101,0.252608,0.207316,0.063806,...,0.672530,0.146222,0.867515,0.686824,0.0,0.496986,0.187443,0.340708,0.143712,0.520549
2241,1,0.026827,0.496253,6,0.543132,0,0.476086,0.351099,0.369118,0.027238,...,0.799256,0.138091,1.000000,0.826505,0.0,0.534956,0.107093,0.447148,0.088639,0.255342
2242,5,0.108490,0.590954,6,0.721398,0,0.483128,0.342678,0.425665,0.110098,...,0.520024,0.115592,0.699501,0.706415,0.0,0.480919,0.189631,0.860649,0.098451,0.795358
